# Actividad 5 · OVARA — Regresión logística vs. Random Forest con validación cruzada y MLflow

**Gestión de Proyectos de IA · Maestría en Inteligencia Artificial (Tecmilenio)** · Emilio Calderón

**Problema:** dada una lesión ovárica vista en ultrasonido y delimitada por su máscara de segmentación, clasificar si es un
**endometrioma** ("quiste de chocolate") o **otra lesión / ovario normal**. Es el primer paso del módulo de apoyo al médico de
OVARA: la segmentación localiza la lesión y este clasificador sugiere su tipo.

**Flujo del notebook**

1. Configuración del entorno (Colab o local)
2. Datos originales y justificación del dataset
3. Limpieza y extracción de características (`fuentes/datos_prep.py`)
4. Exploración rápida
5. Métricas estandarizadas
6. Entrenamiento, ajuste (GridSearchCV + k-fold) y registro en MLflow (`fuentes/train.py`)
7. Resultados y comparación
8. Consulta de los experimentos en MLflow
9. Conclusiones

> Herramienta de apoyo a la investigación. **No es un dispositivo de diagnóstico.**

## 1. Configuración del entorno

El notebook funciona en **Google Colab** y en **local** (VS Code / Jupyter):

* **Colab:** clona el repositorio, instala dependencias y (opcional) descarga MMOTU desde Google Drive para repetir la limpieza.
  Si no se descargan los datos originales, se usa la tabla limpia `datos/datos_limp/mmotu_caracteristicas.csv` incluida en el repositorio.
* **Local:** abrir el notebook desde la carpeta `fuentes/` del proyecto.

Estos modelos son clásicos (scikit-learn): **no requieren GPU**.

In [ ]:
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL   = "https://github.com/emiliocalderonv/TecMilenio_GPAI.git"
SUBCARPETA = "Actividad5"        # carpeta de esta actividad dentro del repositorio
DESCARGAR_MMOTU = False          # True = descargar imágenes originales (173 MB) para repetir la limpieza

if EN_COLAB:
    if not Path("/content/TecMilenio_GPAI").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/TecMilenio_GPAI"], check=True)
    RAIZ = Path("/content/TecMilenio_GPAI") / SUBCARPETA
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(RAIZ / "requirements.txt")], check=True)
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "fuentes" else Path.cwd()

sys.path.insert(0, str(RAIZ / "fuentes"))
os.chdir(RAIZ)
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| Raíz del proyecto:", RAIZ)

In [ ]:
# (Opcional) Descarga de MMOTU original desde Google Drive del autor del dataset
ORIGEN_MMOTU = RAIZ / "datos" / "datos_ini" / "MMOTU" / "OTU_2d"
if DESCARGAR_MMOTU and not ORIGEN_MMOTU.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown
    gdown.download_folder("https://drive.google.com/drive/folders/1c5n0fVKrM9-SZE1kacTXPt1pt844iAs1",
                          output=str(RAIZ / "datos" / "datos_ini" / "MMOTU"), quiet=True)
print("Datos originales de MMOTU disponibles:", ORIGEN_MMOTU.exists())

In [ ]:
import platform, numpy as np, pandas as pd, sklearn, mlflow, matplotlib
import matplotlib.pyplot as plt
print(f"Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__} | MLflow {mlflow.__version__} | {platform.platform()}")

## 2. Datos originales (`datos/datos_ini/`)

| Dataset | Modalidad | Etiquetas | ¿Útil para esta actividad? |
|---|---|---|---|
| **MMOTU** (OTU_2d) | Ultrasonido 2D, 1,469 imágenes | 8 clases por imagen + máscara de la lesión | **Sí.** Tiene clase "quiste de chocolate" (endometrioma) y suficientes casos (336) para clasificación supervisada |
| UT-EndoMRI | Resonancia magnética 3D | Máscaras de útero, ovario, quiste y endometrioma | No para esta tarea: solo ~45 pacientes utilizables con T2 y 13 positivas (Actividad 4). Se reserva para segmentación |
| GLENDA | Laparoscopía, 373 cuadros | Máscaras de 4 localizaciones de endometriosis | No: solo trae imágenes **con** lesión (no hay clase negativa); útil para segmentación |
| ENID | Laparoscopía, 160 cuadros | Máscaras de implantes endometriales | No: una sola clase; útil para detección/segmentación |

Regresión logística y Random Forest trabajan con **datos tabulares**, así que cada imagen de MMOTU se convierte en una fila de
características medibles de la lesión (forma, ecogenicidad, textura y señal Doppler).

## 3. Limpieza y extracción de características (`datos/datos_limp/`)

`fuentes/datos_prep.py` realiza, para cada imagen:

1. **Validación:** imagen y máscara existen y se pueden abrir; clase válida (0–7).
2. **Duplicados:** se descartan imágenes idénticas (hash MD5 de los píxeles).
3. **Consistencia:** si la máscara no coincide en tamaño con la imagen, se reescala (vecino más cercano).
4. **Máscaras inválidas:** se descartan lesiones vacías o menores al 0.1 % de la imagen.
5. **Características (30):** forma (área, circularidad, solidez, ejes…), intensidad dentro de la lesión (media, desviación,
   percentiles, entropía…), contraste con el tejido que la rodea, textura GLCM y fracción de píxeles Doppler.
6. **Valores faltantes / infinitos:** se imputan con la mediana del conjunto de entrenamiento (sin usar prueba).
7. **Variable objetivo:** `endometrioma = 1` si la clase es "quiste de chocolate", si no `0`.

Si los datos originales no están disponibles, se usa la tabla limpia ya generada.

In [ ]:
import json
import datos_prep

DATOS_LIMPIOS = RAIZ / "datos" / "datos_limp" / "mmotu_caracteristicas.csv"
if ORIGEN_MMOTU.exists():
    df = datos_prep.limpiar(ORIGEN_MMOTU, RAIZ / "datos" / "datos_limp", verbose=False)
    print("Limpieza ejecutada a partir de los datos originales.")
else:
    df = pd.read_csv(DATOS_LIMPIOS)
    print("Datos originales no disponibles: se usa la tabla limpia del repositorio.")

reporte = json.loads((RAIZ / "datos" / "datos_limp" / "reporte_limpieza.json").read_text(encoding="utf-8"))
print(json.dumps({k: reporte[k] for k in ("registros_iniciales", "descartes", "correcciones", "registros_finales", "n_caracteristicas")},
                 indent=2, ensure_ascii=False))
df.head()

## 4. Exploración rápida

In [ ]:
resumen = (df.groupby("split").endometrioma.agg(total="count", endometriomas="sum")
             .assign(**{"% endometrioma": lambda t: (100 * t.endometriomas / t.total).round(1)}))
resumen.index = resumen.index.map({"train": "Entrenamiento (CV)", "test": "Prueba (holdout)"})
display(resumen)
display(df.clase_nombre.value_counts().rename("imágenes").to_frame())

In [ ]:
# Visualizaciones exploratorias reproducibles (fuentes/eda.py) → resultados/eda/
import eda
eda.main()
from IPython.display import Image as Img
for g in ("1_distribucion_clases.png", "2_caracteristicas_clave.png", "3_correlaciones.png", "4_valores_atipicos.png"):
    display(Img(filename=str(RAIZ / "resultados" / "eda" / g), width=720))

## 5. Métricas estandarizadas

Las **mismas métricas** se calculan en validación cruzada (media ± desviación de los *k* folds) y en el conjunto de prueba:

| Métrica | Qué mide | Por qué se incluye |
|---|---|---|
| Accuracy | Proporción de aciertos | Referencia general (engañosa con clases desbalanceadas) |
| Balanced accuracy | Promedio del recall de ambas clases | Corrige el desbalance (23 % endometriomas) |
| Precision | De los marcados como endometrioma, cuántos lo eran | Controla falsas alarmas |
| **Recall (sensibilidad)** | De los endometriomas reales, cuántos se detectaron | **Prioritaria clínicamente:** no dejar pasar casos |
| **F1-score** | Media armónica de precision y recall | **Métrica de selección** del mejor hiperparámetro (`refit`) |
| ROC-AUC | Capacidad de separar clases en todos los umbrales | Independiente del umbral |
| PR-AUC | Área bajo la curva precisión–recall | Más informativa que ROC con clases desbalanceadas |

Además se registran **latencia de inferencia** (ms por muestra), *throughput*, **tamaño del modelo** y tiempo de búsqueda.

**Protocolo:** la partición oficial de MMOTU se respeta — *train* (1,000 imágenes) para GridSearchCV con **validación cruzada
estratificada de 5 folds**, y *val* (469) como **conjunto de prueba independiente** que no se usa durante el ajuste.

## 6. Entrenamiento, ajuste y registro en MLflow

`fuentes/train.py` define los dos modelos y sus espacios de búsqueda:

| Modelo | Preprocesamiento | Hiperparámetros explorados | Combinaciones |
|---|---|---|---|
| Regresión logística | Imputación (mediana) + estandarización | `C` ∈ {0.01, 0.1, 1, 10, 100}, penalización ∈ {L1, L2} | 10 |
| Random Forest | Imputación (mediana) | `n_estimators` ∈ {200, 500}, `max_depth` ∈ {None, 8, 16}, `min_samples_leaf` ∈ {1, 3, 5}, `max_features` ∈ {sqrt, 0.3} | 36 |

Ambos usan `class_weight="balanced"` para compensar el desbalance. Cada modelo se registra en MLflow como una corrida con:
parámetros óptimos, métricas de CV y de prueba, latencia, gráficas (matriz de confusión, curvas ROC/PR, importancias), tablas
(`cv_resultados.csv`), el modelo serializado con su firma, y **una corrida anidada por cada combinación del grid**.

In [ ]:
import train
comparacion = train.ejecutar(datos=DATOS_LIMPIOS, folds=5, seed=42)

## 7. Resultados y comparación

In [ ]:
cols = ["cv_f1_media", "cv_f1_std", "cv_roc_auc_media", "test_accuracy", "test_balanced_accuracy", "test_precision",
        "test_recall", "test_f1", "test_roc_auc", "test_pr_auc", "latencia_ms_media", "tamano_modelo_kb", "tiempo_busqueda_s"]
tabla = comparacion[cols].T
tabla.index = ["F1 (CV, media)", "F1 (CV, desv.)", "ROC-AUC (CV)", "Accuracy", "Balanced accuracy", "Precision", "Recall",
               "F1-score", "ROC-AUC", "PR-AUC", "Latencia (ms/muestra)", "Tamaño (KB)", "Tiempo de búsqueda (s)"]
tabla.columns = [c.replace("_", " ") for c in tabla.columns]
display(tabla.style.format("{:.3f}"))
print("Mejores hiperparámetros:")
for m, p in comparacion.mejores_parametros.items():
    print(f"  {m.replace('_', ' ')}: { {k.replace('modelo__', ''): v for k, v in p.items()} }")

In [ ]:
from IPython.display import Image, display as show  # noqa
for m in comparacion.index:
    print(f"\n=== {m.replace('_', ' ')} ===")
    for g in ("matriz_confusion.png", "curvas_roc_pr.png", "importancias.png"):
        show(Image(filename=str(RAIZ / "resultados" / m / g), width=520))

### 7.1 Comparación estadística

`train.py` repite la validación cruzada **5 folds × 10 repeticiones** con los mismos folds para ambos modelos y aplica:
prueba **t corregida de Nadeau-Bengio** (ajusta la dependencia entre folds) y **Wilcoxon**; en el conjunto de prueba,
**McNemar** e **intervalos bootstrap al 95 %** de la diferencia (Random Forest − Regresión logística).

In [ ]:
est = json.loads((RAIZ / "resultados" / "comparacion_estadistica.json").read_text())
filas = [{"Métrica": m.upper(), "Reg. logística (media)": est[m]["media_Regresion_Logistica"],
          "Random Forest (media)": est[m]["media_Random_Forest"], "Diferencia RF − RL": est[m]["diferencia_media"],
          "p (t corregida)": est[m]["p_t_corregida"], "p (Wilcoxon)": est[m]["p_wilcoxon"]} for m in ("f1", "recall", "roc_auc")]
display(pd.DataFrame(filas).set_index("Métrica").style.format("{:.4f}"))
print("Prueba (holdout):", json.dumps(est["prueba"], indent=1))
display(Img(filename=str(RAIZ / "resultados" / "comparacion_estadistica.png"), width=820))

## 8. Consulta de los experimentos en MLflow

Los experimentos se guardan en `mlflow.db` (SQLite) y los artefactos en `mlruns/`, ambos en la raíz del proyecto.

* **Local:** en una terminal dentro de la carpeta del proyecto ejecutar
  `mlflow ui --backend-store-uri sqlite:///mlflow.db` y abrir http://127.0.0.1:5000
* **Colab:** descargar `mlflow.db` y `mlruns/` y abrirlos localmente con el mismo comando, o consultar con la API (celda siguiente).

In [ ]:
runs = mlflow.search_runs(experiment_names=[train.EXPERIMENTO_DEF])
principales = runs[runs["tags.mlflow.parentRunId"].isna()]
print(f"Corridas registradas: {len(runs)} ({len(principales)} principales + {len(runs) - len(principales)} anidadas del grid)")
display(principales[["tags.mlflow.runName", "run_id", "metrics.cv_f1_media", "metrics.test_f1", "metrics.test_recall",
                     "metrics.test_roc_auc", "metrics.latencia_ms_media", "start_time"]]
        .rename(columns=lambda c: c.split(".")[-1]))

## 9. Conclusiones

In [ ]:
c = comparacion
mejor_f1 = c.test_f1.idxmax(); mejor_rec = c.test_recall.idxmax(); mas_rap = c.latencia_ms_media.idxmin()
n = lambda s: s.replace("_", " ")
print(f"• Mejor F1 en prueba: {n(mejor_f1)} ({c.loc[mejor_f1, 'test_f1']:.3f}); mejor recall: {n(mejor_rec)} ({c.loc[mejor_rec, 'test_recall']:.3f}).")
print(f"• ROC-AUC en prueba: " + ", ".join(f"{n(m)} {c.loc[m, 'test_roc_auc']:.3f}" for m in c.index) + ".")
gap = {m: c.loc[m, 'cv_f1_train_media'] - c.loc[m, 'cv_f1_media'] for m in c.index}
print(f"• Brecha F1 entrenamiento–validación (sobreajuste): " + ", ".join(f"{n(m)} {g:.3f}" for m, g in gap.items()) + ".")
print(f"• Más rápido: {n(mas_rap)} ({c.loc[mas_rap, 'latencia_ms_media']:.2f} ms por muestra).")

*(Redacción final de conclusiones en `CHANGELOG.md`.)*